<a href="https://colab.research.google.com/github/watch-duty/radio-transcription/blob/GOO-37-model-eval-improvements/evaluate_transcriptions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Colab to compute metrics from a JSON manifest with transcriptions from various models.

The google drive version of this colab (for iterating and editing) [is here](https://colab.research.google.com/drive/1HJs9lUMsg5dyK4uwuLmzxzbtKEE9tkbi?authuser=1#scrollTo=72af-Ue2_BwC). Once you have a unit of edit completed, you can check-in the drive version into [this github directory](https://github.com/watch-duty/radio-transcription/tree/main/model/colabs).

In [ ]:
!pip install evaluate torch
!pip install bert_score
!pip install nemo_toolkit['asr']
!pip install nemo_text_processing

In [ ]:
from google.colab import files

print("Please select the file 'playground_parakeet_and_canary_flash.json' from your local machine.")
uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')

# After uploading, the file will be in the current working directory, e.g., '/content/playground_parakeet_and_canary_flash.json'
# You might need to update the `inference_results` variable accordingly.
# For example, if the uploaded file name is exactly 'playground_parakeet_and_canary_flash.json':
# inference_results = '/content/playground_parakeet_and_canary_flash.json'

In [ ]:
#@title Imports
import json
import os
import evaluate # Import the evaluate library
from nemo.collections.asr.parts.utils import transcribe_utils
from nemo.collections.asr.metrics.wer import word_error_rate
# Removed problematic ITN import: import nemo.collections.asr.parts.utils.InverseTextNormalizer as InverseNormalizer

inference_results = "/content/playground_parakeet_and_canary_flash.json"  #@param
postfixes_to_eval = ['parakeet-tdt-06b-v2', 'canary-1b-flash']  #@param
eval_output = "/tmp/playground_parakeet_and_canary_flash_eval.json"

# Load the BERTScore metric
bertscore = evaluate.load("bertscore")

# Removed problematic ITN initialization: itn_processor = InverseNormalizer(lang='en')

In [ ]:

#@title Imports
import json
import os
import evaluate # Import the evaluate library
from nemo.collections.asr.parts.utils import transcribe_utils
from nemo.collections.asr.metrics.wer import word_error_rate
from nemo_text_processing.inverse_text_normalization.inverse_normalize import InverseNormalizer
import pandas as pd

inference_results = "/content/playground_parakeet_and_canary_flash.json"  #@param
postfixes_to_eval = ['parakeet-tdt-06b-v2', 'canary-1b-flash']  #@param
eval_output = "/tmp/playground_parakeet_and_canary_flash_eval.json"

itn_processor = InverseNormalizer(lang='en')

# Load the BERTScore metric
bertscore = evaluate.load("bertscore")

In [ ]:
import json
import os
import evaluate # Import the evaluate library
from nemo.collections.asr.parts.utils import transcribe_utils
from nemo.collections.asr.metrics.wer import word_error_rate

def _process_single_row(
    data_row,
    gt_text_field,
    prediction_field_prefix,
    itn_processor,
    pc_processor,
    postfixes_to_eval,
    bertscore
):
  """Processes a single data row to calculate WER and BERTScore."""
  audio_filepath = data_row["audio_filepath"]
  ground_truth_text_raw = data_row.get(gt_text_field, "")

  # Remove <unknown> token from ground_truth_text_raw
  ground_truth_text_raw = ground_truth_text_raw.replace("<unknown>", "").strip()

  if not ground_truth_text_raw:
    print(f"Skipping example: {audio_filepath}, ground truth is empty or only <unknown> after processing")
    return None, None, None, True # Indicate skipped row

  # Apply ITN to ground truth text, this converts numbers to string representation.
  ground_truth_text_itn = [itn_processor.inverse_normalize(text=ground_truth_text_raw, verbose=False)]

  # Check if all models have a prediction for this row
  for postfix in postfixes_to_eval:
    pred_text_field = prediction_field_prefix + postfix
    predicted_text_value = data_row.get(pred_text_field)
    if predicted_text_value is None or predicted_text_value == "":
      print(f"Skipping example: {audio_filepath}, missing or empty prediction for model: {postfix}")
      return None, None, None, True # Indicate skipped row

  # Process ITN'd ground truth text for WER
  ground_truth_text_processed = pc_processor.do_lowercase(ground_truth_text_itn)
  ground_truth_text_processed = pc_processor.rm_punctuation(ground_truth_text_processed)

  output_row = data_row.copy()
  # Add processed ground truth to output_row
  output_row['ground_truth_processed'] = ground_truth_text_processed[0]

  predictions_processed_by_postfix = {}
  for postfix in postfixes_to_eval:
    pred_text_field = prediction_field_prefix + postfix
    predicted_text_raw = data_row.get(pred_text_field)

    # Apply ITN to predicted text
    predicted_text_itn = [itn_processor.inverse_normalize(text=predicted_text_raw, verbose=False)]

    # Process ITN'd predicted text for WER
    predicted_text_processed = pc_processor.do_lowercase(predicted_text_itn)
    predicted_text_processed = pc_processor.rm_punctuation(predicted_text_processed)
    predictions_processed_by_postfix[postfix] = predicted_text_processed

    sample_wer = word_error_rate(hypotheses=predicted_text_processed, references=ground_truth_text_processed, use_cer=False)
    sample_wer = round(100 * sample_wer, 2)
    output_row["wer_" + postfix] = sample_wer

    # Calculate BERTScore using ITN'd texts
    bertscore_results = bertscore.compute(predictions=predicted_text_itn, references=ground_truth_text_itn, lang="en")
    output_row[f"bertscore_f1_{postfix}"] = round(bertscore_results['f1'][0], 4)
    output_row[f"bertscore_precision_{postfix}"] = round(bertscore_results['precision'][0], 4)
    output_row[f"bertscore_recall_{postfix}"] = round(bertscore_results['recall'][0], 4)

    # Add processed predicted text to output_row
    output_row[f'pred_text_processed_{postfix}'] = predicted_text_processed[0]

  return output_row, ground_truth_text_processed, predictions_processed_by_postfix, False

def _calculate_and_print_aggregate_metrics(all_ground_truths, all_predictions_by_postfix, postfixes_to_evaluate, output_json):
  """Calculates and prints aggregate WER and BERTScore metrics."""
  print("\n--- Aggregate Metrics ---")
  for postfix in postfixes_to_evaluate:
    dataset_wer = word_error_rate(hypotheses=all_predictions_by_postfix[postfix], references=all_ground_truths, use_cer=False)
    dataset_wer = round(100 * dataset_wer, 2)
    print(f"Dataset WER for {postfix}: {dataset_wer:.1f}")

    # Aggregate BERTScore (mean of all F1 scores)
    all_f1_scores = []
    with open(output_json, 'r', encoding='utf-8') as temp_f:
        for line in temp_f:
            row = json.loads(line)
            if f"bertscore_f1_{postfix}" in row:
                all_f1_scores.append(row[f"bertscore_f1_{postfix}"])
    if all_f1_scores:
        avg_f1 = sum(all_f1_scores) / len(all_f1_scores)
        print(f"Dataset BERTScore F1 for {postfix}: {avg_f1:.4f}")

def run_evaluation(input_json, output_json, postfixes_to_evaluate):

  gt_text_field = "text"
  prediction_field_prefix = "pred_text_"
  pc_processor = transcribe_utils.PunctuationCapitalization(".,?")

  total_rows = 0
  skipped_rows = 0
  evaluated_rows = 0

  all_ground_truths = []
  all_predictions_by_postfix = {postfix: [] for postfix in postfixes_to_evaluate}

  with open(input_json, 'r', encoding='utf-8') as f_in, \
       open(output_json, 'w', encoding='utf-8') as f_out:

    for line in f_in:
      total_rows += 1
      data_row = json.loads(line)

      output_row, gt_processed, preds_processed_by_postfix, was_skipped = \
          _process_single_row(data_row, gt_text_field, prediction_field_prefix,
                              itn_processor, pc_processor, postfixes_to_evaluate, bertscore)

      if was_skipped:
        skipped_rows += 1
        continue

      evaluated_rows += 1
      all_ground_truths.extend(gt_processed)
      for postfix, preds in preds_processed_by_postfix.items():
          all_predictions_by_postfix[postfix].extend(preds)

      f_out.write(json.dumps(output_row) + '\n')

  _calculate_and_print_aggregate_metrics(all_ground_truths, all_predictions_by_postfix, postfixes_to_evaluate, output_json)

  print(f"\nSuccessfully evaluated {input_json}\nWrote per sample evaluation to: {output_json}")
  print(f"\n--- Row Statistics ---")
  print(f"Total rows in manifest: {total_rows}")
  print(f"Skipped rows (missing ground truth or prediction): {skipped_rows}")
  print(f"Evaluated rows: {evaluated_rows}")


In [ ]:
run_evaluation(inference_results, eval_output, postfixes_to_eval)

In [ ]:
#@title  Visualize results

# Define a styling function for BERTScore F1 values
def highlight_bertscore(s):
    # s is a Series representing a row
    # Return a Series of style strings for each cell in the row
    styles = [''] * len(s)
    for i, (col_name, value) in enumerate(s.items()):
        if col_name.startswith('bertscore_f1_'):
            if pd.notna(value):
                if value >= 0.9:
                    styles[i] = 'background-color: #d4edda' # Light green
                elif value >= 0.7:
                    styles[i] = 'background-color: #fff3cd' # Light yellow
                else: # value < 0.7
                    styles[i] = 'background-color: #f8d7da' # Light red
    return styles

# Define a styling function for WER values
def highlight_wer(s):
    # s is a Series representing a row
    # Return a Series of style strings for each cell in the row
    styles = [''] * len(s)
    for i, (col_name, value) in enumerate(s.items()):
        if col_name.startswith('wer_'):
            if pd.notna(value):
                if value < 5:  # Excellent
                    styles[i] = 'background-color: #d4edda' # Light green
                elif value < 30: # Moderate
                    styles[i] = 'background-color: #fff3cd' # Light yellow
                else: # Poor
                    styles[i] = 'background-color: #f8d7da' # Light red
    return styles

# Load the evaluation output JSON into a DataFrame
df_eval = pd.read_json(eval_output, lines=True)

# Prepare columns for display
display_columns = ['audio_filepath', 'text', 'ground_truth_processed']

# These lists are used for calculating overall metrics, not directly for display here
bertscore_f1_cols = []
wer_cols = []

# Populate all available WER and BERTScore F1 columns for overall calculations
for postfix in postfixes_to_eval:
    if f'wer_{postfix}' in df_eval.columns:
        wer_cols.append(f'wer_{postfix}')
    if f'bertscore_f1_{postfix}' in df_eval.columns:
        bertscore_f1_cols.append(f'bertscore_f1_{postfix}')

# Add all relevant columns (pred_text, WER, BERTScore F1, and processed pred_text) for each model to display_columns
for postfix in postfixes_to_eval:
    if f'pred_text_{postfix}' in df_eval.columns:
        display_columns.append(f'pred_text_{postfix}')
    if f'pred_text_processed_{postfix}' in df_eval.columns:
        display_columns.append(f'pred_text_processed_{postfix}')
    if f'wer_{postfix}' in df_eval.columns:
        display_columns.append(f'wer_{postfix}')
    if f'bertscore_f1_{postfix}' in df_eval.columns:
        display_columns.append(f'bertscore_f1_{postfix}')

# Calculate overall BERTScore F1 for sorting (mean across all models)
df_eval['overall_bertscore_f1'] = df_eval[bertscore_f1_cols].mean(axis=1)

# Display examples with low overall BERTScore F1
print("\n--- Examples with Low Overall BERTScore (F1) Across All Models ---")
display(df_eval.sort_values(by='overall_bertscore_f1', ascending=True)[display_columns + ['overall_bertscore_f1']].head(10).style.apply(highlight_bertscore, axis=1).apply(highlight_wer, axis=1))

# Display examples with high overall BERTScore F1
print("\n--- Examples with High Overall BERTScore (F1) Across All Models ---")
display(df_eval.sort_values(by='overall_bertscore_f1', ascending=False)[display_columns + ['overall_bertscore_f1']].head(10).style.apply(highlight_bertscore, axis=1).apply(highlight_wer, axis=1))

# Calculate overall WER for sorting (mean across all models)
df_eval['overall_wer'] = df_eval[wer_cols].mean(axis=1)

# Display examples with high overall WER Across All Models
print("\n--- Examples with High Overall WER Across All Models ---")
display(df_eval.sort_values(by='overall_wer', ascending=False)[display_columns + ['overall_wer']].head(10).style.apply(highlight_bertscore, axis=1).apply(highlight_wer, axis=1))

# Display examples with low overall WER Across All Models
print("\n--- Examples with Low Overall WER Across All Models ---")
display(df_eval.sort_values(by='overall_wer', ascending=True)[display_columns + ['overall_wer']].head(10).style.apply(highlight_bertscore, axis=1).apply(highlight_wer, axis=1))

In [ ]:

# @title Investigate indiviual audio file
specific_audio_filename = 'ca_grass_valley2F202508202FCDF_Cmd_2_20250820_15' #@param string

# Filter the DataFrame for the specific audio file, checking if the filename is contained in the full path
rows_to_examine = df_eval[df_eval['audio_filepath'].str.contains(specific_audio_filename, na=False)]

if not rows_to_examine.empty:
    print(f"Transcriptions for audio_filepath containing: {specific_audio_filename}")
    display(rows_to_examine.sort_values(by='offset', ascending=True)[[
        'audio_filepath',
        'text',
        'offset',
        'duration',
        'wer_parakeet-tdt-06b-v2',
        'bertscore_f1_parakeet-tdt-06b-v2',
        'wer_canary-1b-flash',
        'bertscore_f1_canary-1b-flash',
        'pred_text_parakeet-tdt-06b-v2',
        'pred_text_canary-1b-flash'
    ]])
else:
    print(f"No rows found for audio_filepath containing '{specific_audio_filename}'.")

### Average WER and BERTScore F1 Per Audio Filepath


In [ ]:
pd.set_option('display.max_colwidth', None)

avg_metrics_per_audio = df_eval.groupby('audio_filepath').agg(
    mean_wer_parakeet=('wer_parakeet-tdt-06b-v2', 'mean'),
    mean_bertscore_f1_parakeet=('bertscore_f1_parakeet-tdt-06b-v2', 'mean'),
    mean_wer_canary=('wer_canary-1b-flash', 'mean'),
    mean_bertscore_f1_canary=('bertscore_f1_canary-1b-flash', 'mean')
).reset_index()

print("\n--- Average WER and BERTScore F1 per Audio Filepath ---")
display(avg_metrics_per_audio.sort_values(by='mean_wer_parakeet', ascending=False))

In [ ]:
#@title Visualize WER x BertScore per audio file
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

# Create a copy to avoid modifying the original DataFrame unintentionally
plot_df = avg_metrics_per_audio.copy()

# Extract short audio_filepath names for better plotting
plot_df['short_audio_filepath'] = plot_df['audio_filepath'].apply(lambda x: os.path.basename(x))

# Prepare data for Average WER plot
wer_plot_data = plot_df[['short_audio_filepath', 'mean_wer_parakeet', 'mean_wer_canary']]
wer_plot_data_melted = wer_plot_data.melt(
    id_vars='short_audio_filepath',
    var_name='Model',
    value_name='Average WER'
)
wer_plot_data_melted['Model'] = wer_plot_data_melted['Model'].replace({
    'mean_wer_parakeet': 'Parakeet-tdt-06b-v2',
    'mean_wer_canary': 'Canary-1b-flash'
})

# Prepare data for Average BERTScore F1 plot
bertscore_plot_data = plot_df[['short_audio_filepath', 'mean_bertscore_f1_parakeet', 'mean_bertscore_f1_canary']]
bertscore_plot_data_melted = bertscore_plot_data.melt(
    id_vars='short_audio_filepath',
    var_name='Model',
    value_name='Average BERTScore F1'
)
bertscore_plot_data_melted['Model'] = bertscore_plot_data_melted['Model'].replace({
    'mean_bertscore_f1_parakeet': 'Parakeet-tdt-06b-v2',
    'mean_bertscore_f1_canary': 'Canary-1b-flash'
})

# Create subplots
fig, axes = plt.subplots(1, 2, figsize=(20, 8), sharey=False) # sharey=False because WER and BERTScore F1 scales are different

# Plot Average WER
sns.barplot(x='short_audio_filepath', y='Average WER', hue='Model', data=wer_plot_data_melted, ax=axes[0], palette='viridis')
axes[0].set_title('Average Word Error Rate (WER) per Audio Filepath')
axes[0].set_ylabel('Average WER (%)')
axes[0].set_xlabel('Audio Filepath')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

# Plot Average BERTScore F1
sns.barplot(x='short_audio_filepath', y='Average BERTScore F1', hue='Model', data=bertscore_plot_data_melted, ax=axes[1], palette='plasma')
axes[1].set_title('Average BERTScore F1 per Audio Filepath')
axes[1].set_ylabel('Average BERTScore F1')
axes[1].set_xlabel('Audio Filepath')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()